# BraTS 2024 Glioma MRI Preprocessing Pipeline

This notebook develops a reproducible preprocessing pipeline for the
BraTS 2024 Adult Glioma dataset in preparation for 3D deep-learning
segmentation.

The preceding exploratory notebook examined individual NIfTI volumes,
multimodal MRI alignment, reference tumour segmentations, MRI intensity
distributions and modality-wise normalization.

This notebook extends those concepts to the complete dataset using MONAI.

## Objectives

1. Discover all available BraTS training cases.
2. Construct a dataset manifest linking each case to its four MRI
   modalities and reference segmentation.
3. Validate dataset completeness before model training.
4. Define reproducible training and validation partitions.
5. Load multimodal NIfTI data using MONAI.
6. Apply spatial and intensity preprocessing.
7. Prepare segmentation targets for supervised learning.
8. Develop an appropriate 3D cropping and patch-sampling strategy.
9. Introduce training-specific data augmentation.
10. Construct MONAI datasets and PyTorch DataLoaders.
11. Inspect model-ready batches before beginning model training.

## 1. Environment and Dataset Configuration

The preprocessing pipeline begins by importing the required libraries and
defining the locations of the project repository and externally stored
BraTS dataset.

The BraTS imaging data are intentionally stored outside the Git repository
and are not version-controlled.

In [1]:
# -------------------------------------------------------------
# IMPORT LIBRARIES
# -------------------------------------------------------------
#
# Standard Python and scientific-computing libraries are used
# alongside MONAI, which provides medical-imaging-specific
# preprocessing and deep-learning utilities.

from pathlib import Path

import numpy as np
import torch
import monai

print("NumPy version:", np.__version__)
print("PyTorch version:", torch.__version__)
print("MONAI version:", monai.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

NumPy version: 2.4.6
PyTorch version: 2.14.1+cu130
MONAI version: 1.6.1
CUDA available: True
GPU: NVIDIA GeForce RTX 4080 SUPER


In [2]:
# -------------------------------------------------------------
# DEFINE PROJECT AND DATASET PATHS
# -------------------------------------------------------------
#
# PROJECT_ROOT points to the Git repository.
#
# DATA_ROOT points to the externally stored BraTS dataset.
# The medical imaging dataset is deliberately kept outside the
# repository so that large source data are never committed to Git.

PROJECT_ROOT = Path(
    "/mnt/c/Users/USER/OneDrive/Documents/GitHub/glioma-mri-segmentation"
)

DATA_ROOT = Path(
    "/home/corvus_splendens/datasets/brats2024-gli"
)

TRAINING_DATA_ROOT = (
    DATA_ROOT
    / "training"
    / "training_data1_v2"
)

print("Project root exists:", PROJECT_ROOT.exists())
print("Dataset root exists:", DATA_ROOT.exists())
print("Training data exists:", TRAINING_DATA_ROOT.exists())

Project root exists: True
Dataset root exists: True
Training data exists: True


## 2. Dataset Discovery

Each BraTS training case is stored in a separate directory identified by a
BraTS case identifier.

The available case directories are discovered programmatically rather than
manually enumerated. Sorting the directories ensures deterministic ordering
before subsequent dataset partitioning.

In [3]:
# -------------------------------------------------------------
# DISCOVER BRATS TRAINING CASES
# -------------------------------------------------------------
#
# Identify all directories corresponding to BraTS glioma cases.
#
# Sorting ensures that the initial ordering is deterministic
# rather than dependent on filesystem ordering.

case_directories = sorted(
    path
    for path in TRAINING_DATA_ROOT.iterdir()
    if path.is_dir()
    and path.name.startswith("BraTS-GLI-")
)

print(
    "Number of BraTS training cases:",
    len(case_directories)
)

print("\nFirst five cases:")

for case_directory in case_directories[:5]:
    print(case_directory.name)

Number of BraTS training cases: 1350

First five cases:
BraTS-GLI-00005-100
BraTS-GLI-00005-101
BraTS-GLI-00006-100
BraTS-GLI-00006-101
BraTS-GLI-00008-100


## 3. Dataset Manifest

Each BraTS case contains four spatially aligned MRI modalities and one
reference segmentation.

For each case, a dataset entry is created containing:

- `image` — paths to the T1n, T1c, T2w and T2-FLAIR volumes;
- `label` — path to the corresponding reference segmentation;
- `case_id` — the BraTS case identifier.

The four MRI paths are stored together so that MONAI can subsequently load
them as channels of a single multimodal input.

Constructing the dataset programmatically reduces the risk of manually
mismatching MRI volumes and segmentation labels between patients.

In [ ]:
# -------------------------------------------------------------
# BUILD THE BRATS DATASET MANIFEST
# -------------------------------------------------------------
#
# Each dictionary represents one BraTS case.
#
# "image" contains the four MRI modalities in a fixed order:
#
#     0 = T1n
#     1 = T1c
#     2 = T2w
#     3 = T2-FLAIR
#
# "label" contains the corresponding reference segmentation.
#
# Keeping the modality order fixed is essential because the neural network will eventually interpret these as four distinct input channels.

dataset_manifest = []

for case_directory in case_directories:

    case_id = case_directory.name

    image_paths = [
        case_directory / f"{case_id}-t1n.nii.gz",
        case_directory / f"{case_id}-t1c.nii.gz",
        case_directory / f"{case_id}-t2w.nii.gz",
        case_directory / f"{case_id}-t2f.nii.gz",
    ]

    segmentation_path = (
        case_directory / f"{case_id}-seg.nii.gz"
    )

    dataset_manifest.append(
        {
            "case_id": case_id,
            "image": [
                str(path)
                for path in image_paths
            ],
            "label": str(segmentation_path),
        }
    )


print(
    "Number of cases in manifest:",
    len(dataset_manifest)
)

print("\nFirst manifest entry:")
print(dataset_manifest[0])

Number of cases in manifest: 1350

First manifest entry:
{'case_id': 'BraTS-GLI-00005-100', 'image': ['/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t1n.nii.gz', '/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t1c.nii.gz', '/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t2w.nii.gz', '/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-t2f.nii.gz'], 'label': '/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00005-100/BraTS-GLI-00005-100-seg.nii.gz'}


## 4. Dataset Integrity Validation

Before constructing the preprocessing pipeline, the dataset manifest is
validated to confirm that every expected MRI volume and reference
segmentation exists.

For each BraTS case, five files are expected:

- four MRI modalities (`t1n`, `t1c`, `t2w`, `t2f`);
- one reference segmentation (`seg`).

With 1,350 training cases, the complete dataset should therefore contain
6,750 expected NIfTI files.

Performing this validation before model training helps detect missing or
incorrectly named files before they cause failures later in the pipeline.

In [5]:
# -------------------------------------------------------------
# VALIDATE DATASET FILE INTEGRITY
# -------------------------------------------------------------
#
# The manifest contains paths that we constructed from the
# expected BraTS naming convention.
#
# We now verify that every one of those files actually exists
# on disk before building the MONAI preprocessing pipeline.

missing_files = []

total_expected_files = 0


for case in dataset_manifest:

    # Each case should contain four MRI files.
    for image_path in case["image"]:

        total_expected_files += 1

        if not Path(image_path).is_file():
            missing_files.append(image_path)

    # Each case should also contain one reference segmentation.
    total_expected_files += 1

    if not Path(case["label"]).is_file():
        missing_files.append(case["label"])


print("Cases checked:", len(dataset_manifest))

print(
    "Expected files checked:",
    total_expected_files
)

print(
    "Missing files:",
    len(missing_files)
)


if missing_files:

    print("\nFirst missing files:")

    for path in missing_files[:10]:
        print(path)

else:

    print(
        "\nDataset integrity check passed: "
        "all expected files are present."
    )

Cases checked: 1350
Expected files checked: 6750
Missing files: 0

Dataset integrity check passed: all expected files are present.


## 4. Dataset Integrity Validation

The dataset manifest contains the expected file paths for each BraTS case,
but constructing a path does not confirm that the corresponding file
actually exists.

Before developing the MONAI preprocessing pipeline, every manifest entry is
therefore checked to confirm the presence of all four MRI modalities and the
corresponding reference segmentation.

With 1,350 cases and five expected NIfTI files per case, the complete
manifest should reference 6,750 existing files.

In [6]:
# -------------------------------------------------------------
# VALIDATE DATASET FILE INTEGRITY
# -------------------------------------------------------------
#
# The dataset manifest contains paths constructed according to
# the expected BraTS naming convention.
#
# This check confirms that every path in the manifest actually
# points to a file that exists on disk.

missing_files = []

# Keep track of how many files we have checked.
total_expected_files = 0


for case in dataset_manifest:

    # ---------------------------------------------------------
    # CHECK THE FOUR MRI MODALITIES
    # ---------------------------------------------------------
    #
    # case["image"] contains the paths to:
    # T1n, T1c, T2w and T2-FLAIR.

    for image_path in case["image"]:

        total_expected_files += 1

        # Path(...).is_file() returns True only if the path
        # exists and points to a file.
        if not Path(image_path).is_file():
            missing_files.append(image_path)


    # ---------------------------------------------------------
    # CHECK THE REFERENCE SEGMENTATION
    # ---------------------------------------------------------

    total_expected_files += 1

    if not Path(case["label"]).is_file():
        missing_files.append(case["label"])


# -------------------------------------------------------------
# REPORT THE RESULTS
# -------------------------------------------------------------

print(
    "Cases checked:",
    len(dataset_manifest)
)

print(
    "Expected files checked:",
    total_expected_files
)

print(
    "Missing files:",
    len(missing_files)
)


# If any files are missing, display the first ten so that the
# problem can be investigated without printing an enormous list.
if missing_files:

    print("\nFirst missing files:")

    for path in missing_files[:10]:
        print(path)

else:

    print(
        "\nDataset integrity check passed: "
        "all expected files are present."
    )

Cases checked: 1350
Expected files checked: 6750
Missing files: 0

Dataset integrity check passed: all expected files are present.


## 5. Patient-Level Dataset Structure

The BraTS dataset may contain multiple imaging timepoints from the same
patient. Cases sharing the same core BraTS subject identifier therefore
must not be distributed across different training and evaluation subsets.

For example:

- `BraTS-GLI-00005-100`
- `BraTS-GLI-00005-101`

represent different imaging timepoints associated with the same underlying
subject identifier:

`BraTS-GLI-00005`

Dataset partitioning will therefore be performed at the patient level
rather than the individual scan level to prevent data leakage between
training and evaluation datasets.

In [7]:
# -------------------------------------------------------------
# IDENTIFY UNIQUE BRATS SUBJECTS
# -------------------------------------------------------------
#
# A BraTS case identifier contains both:
#
#     subject identifier + imaging timepoint
#
# Example:
#
#     BraTS-GLI-00005-100
#     └─ BraTS-GLI-00005 = subject
#                       └─ 100 = timepoint
#
# Multiple timepoints belonging to the same subject must remain
# together when the dataset is later divided into training and
# evaluation subsets.

for case in dataset_manifest:

    case_id = case["case_id"]

    # Remove the final "-XXX" timepoint component.
    subject_id = case_id.rsplit("-", 1)[0]

    # Store the subject identifier directly in the manifest.
    case["subject_id"] = subject_id


unique_subject_ids = sorted(
    {
        case["subject_id"]
        for case in dataset_manifest
    }
)


print(
    "Number of imaging cases:",
    len(dataset_manifest)
)

print(
    "Number of unique subjects:",
    len(unique_subject_ids)
)

print(
    "Cases from first five subjects:"
)

for subject_id in unique_subject_ids[:5]:

    subject_cases = [
        case["case_id"]
        for case in dataset_manifest
        if case["subject_id"] == subject_id
    ]

    print(
        subject_id,
        "->",
        subject_cases
    )

Number of imaging cases: 1350
Number of unique subjects: 613
Cases from first five subjects:
BraTS-GLI-00005 -> ['BraTS-GLI-00005-100', 'BraTS-GLI-00005-101']
BraTS-GLI-00006 -> ['BraTS-GLI-00006-100', 'BraTS-GLI-00006-101']
BraTS-GLI-00008 -> ['BraTS-GLI-00008-100', 'BraTS-GLI-00008-101', 'BraTS-GLI-00008-102', 'BraTS-GLI-00008-103']
BraTS-GLI-00009 -> ['BraTS-GLI-00009-100', 'BraTS-GLI-00009-101']
BraTS-GLI-00020 -> ['BraTS-GLI-00020-100', 'BraTS-GLI-00020-101']


In [8]:
# -------------------------------------------------------------
# EXAMINE THE NUMBER OF IMAGING CASES PER SUBJECT
# -------------------------------------------------------------
#
# The dataset contains multiple imaging cases for many subjects.
# We count how many cases belong to each subject so that the
# longitudinal/repeated-imaging structure of the dataset is
# explicit before performing patient-level partitioning.

cases_per_subject = {}

for case in dataset_manifest:

    subject_id = case["subject_id"]

    if subject_id not in cases_per_subject:
        cases_per_subject[subject_id] = 0

    cases_per_subject[subject_id] += 1


# Count how many subjects have 1, 2, 3, ... imaging cases.
case_count_distribution = {}

for number_of_cases in cases_per_subject.values():

    if number_of_cases not in case_count_distribution:
        case_count_distribution[number_of_cases] = 0

    case_count_distribution[number_of_cases] += 1


print("Imaging cases per subject:")

for number_of_cases in sorted(case_count_distribution):

    print(
        f"  {number_of_cases} case(s): "
        f"{case_count_distribution[number_of_cases]} subjects"
    )


print(
    "\nMaximum cases for a single subject:",
    max(cases_per_subject.values())
)

Imaging cases per subject:
  1 case(s): 129 subjects
  2 case(s): 375 subjects
  3 case(s): 39 subjects
  4 case(s): 30 subjects
  5 case(s): 21 subjects
  6 case(s): 10 subjects
  7 case(s): 5 subjects
  8 case(s): 3 subjects
  10 case(s): 1 subjects

Maximum cases for a single subject: 10


## 6. Patient-Level Train/Validation/Test Split

The 1,350 imaging cases originate from 613 unique subjects, with many
subjects contributing multiple imaging timepoints.

To prevent patient-level data leakage, dataset partitioning is therefore
performed using the unique subject identifiers rather than individual MRI
cases.

Subjects are divided into:

- **70% training** — used to optimize model parameters;
- **15% validation** — used for model selection and development;
- **15% held-out test** — reserved for final evaluation.

All imaging cases belonging to a given subject remain within the same
partition.

A fixed random seed is used to make the partition reproducible.

In [10]:
# -------------------------------------------------------------
# CREATE PATIENT-LEVEL TRAIN / VALIDATION / TEST SPLITS
# -------------------------------------------------------------
#
# IMPORTANT:
#
# We split SUBJECTS rather than individual imaging cases.
#
# This ensures that multiple scans belonging to the same patient
# cannot appear in different dataset partitions.

from sklearn.model_selection import train_test_split


# -------------------------------------------------------------
# STEP 1: SPLIT SUBJECTS INTO TRAINING AND TEMPORARY SETS
# -------------------------------------------------------------
#
# 70% of subjects will be assigned to training.
#
# The remaining 30% will temporarily be kept together and then
# divided equally into validation and test subjects.

train_subjects, temporary_subjects = train_test_split(
    unique_subject_ids,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)


# -------------------------------------------------------------
# STEP 2: DIVIDE THE REMAINING 30% EQUALLY
# -------------------------------------------------------------
#
# Half of the temporary subjects become validation subjects.
# The other half become held-out test subjects.
#
# This gives approximately:
#
#     70% training
#     15% validation
#     15% test

validation_subjects, test_subjects = train_test_split(
    temporary_subjects,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)


# Convert the subject lists into sets.
#
# Sets make membership checks efficient and will also make it
# easy to test for overlap between the partitions later.

train_subjects = set(train_subjects)
validation_subjects = set(validation_subjects)
test_subjects = set(test_subjects)


print(
    "Training subjects:",
    len(train_subjects)
)

print(
    "Validation subjects:",
    len(validation_subjects)
)

print(
    "Test subjects:",
    len(test_subjects)
)

print(
    "\nTotal subjects:",
    (
        len(train_subjects)
        + len(validation_subjects)
        + len(test_subjects)
    )
)

Training subjects: 429
Validation subjects: 92
Test subjects: 92

Total subjects: 613


## 7. Assign Imaging Cases to Dataset Partitions

The preceding split assigns each unique subject to the training,
validation or held-out test cohort.

The subject-level assignments are now mapped back to the dataset manifest
so that every imaging case belonging to a subject remains within the same
partition.

The resulting case-level manifests will subsequently be supplied to the
MONAI preprocessing and data-loading pipeline.

In [11]:
# -------------------------------------------------------------
# ASSIGN IMAGING CASES TO THEIR PATIENT-LEVEL PARTITIONS
# -------------------------------------------------------------
#
# The previous step divided SUBJECTS into training, validation
# and test groups.
#
# We now use those subject assignments to divide the 1,350
# imaging cases in the dataset manifest.

train_cases = []
validation_cases = []
test_cases = []


for case in dataset_manifest:

    subject_id = case["subject_id"]

    # If this case belongs to a training subject, place the
    # entire imaging case into the training manifest.
    if subject_id in train_subjects:
        train_cases.append(case)

    # Otherwise, check whether it belongs to a validation subject.
    elif subject_id in validation_subjects:
        validation_cases.append(case)

    # Otherwise, check whether it belongs to a test subject.
    elif subject_id in test_subjects:
        test_cases.append(case)

    # Every subject should belong to exactly one partition.
    # Reaching this branch would indicate an error in our split.
    else:
        raise ValueError(
            f"Subject {subject_id} was not assigned "
            "to any dataset partition."
        )


print("Training imaging cases:", len(train_cases))
print("Validation imaging cases:", len(validation_cases))
print("Test imaging cases:", len(test_cases))

print(
    "\nTotal imaging cases:",
    len(train_cases)
    + len(validation_cases)
    + len(test_cases)
)

Training imaging cases: 936
Validation imaging cases: 216
Test imaging cases: 198

Total imaging cases: 1350


## 8. Verify Patient-Level Separation

Because multiple imaging cases may originate from the same subject, the
dataset partitions are explicitly checked for patient-level overlap.

A valid partition requires:

- no subject to appear in both training and validation;
- no subject to appear in both training and test;
- no subject to appear in both validation and test;
- all 613 subjects and all 1,350 imaging cases to remain accounted for.

These checks help ensure that model performance is not artificially
inflated by evaluating the model on additional scans from patients that
were already encountered during training.

In [12]:
# -------------------------------------------------------------
# VERIFY PATIENT-LEVEL DATASET SEPARATION
# -------------------------------------------------------------
#
# We explicitly check that no subject occurs in more than one
# dataset partition.
#
# The intersection of two sets contains values present in BOTH.
# For a leakage-free split, every intersection should be empty.

train_validation_overlap = (
    train_subjects.intersection(validation_subjects)
)

train_test_overlap = (
    train_subjects.intersection(test_subjects)
)

validation_test_overlap = (
    validation_subjects.intersection(test_subjects)
)


print(
    "Train / validation subject overlap:",
    len(train_validation_overlap)
)

print(
    "Train / test subject overlap:",
    len(train_test_overlap)
)

print(
    "Validation / test subject overlap:",
    len(validation_test_overlap)
)


# -------------------------------------------------------------
# VERIFY THAT NOTHING WAS LOST DURING PARTITIONING
# -------------------------------------------------------------

total_partitioned_subjects = (
    len(train_subjects)
    + len(validation_subjects)
    + len(test_subjects)
)

total_partitioned_cases = (
    len(train_cases)
    + len(validation_cases)
    + len(test_cases)
)


print(
    "\nSubjects accounted for:",
    total_partitioned_subjects,
    "/",
    len(unique_subject_ids)
)

print(
    "Imaging cases accounted for:",
    total_partitioned_cases,
    "/",
    len(dataset_manifest)
)


# -------------------------------------------------------------
# AUTOMATED SAFETY CHECKS
# -------------------------------------------------------------
#
# assert means:
#
#     "This condition MUST be true."
#
# If it is not true, Python stops execution with an error.

assert len(train_validation_overlap) == 0
assert len(train_test_overlap) == 0
assert len(validation_test_overlap) == 0

assert total_partitioned_subjects == len(unique_subject_ids)
assert total_partitioned_cases == len(dataset_manifest)


print(
    "\nPatient-level split validation passed."
)

Train / validation subject overlap: 0
Train / test subject overlap: 0
Validation / test subject overlap: 0

Subjects accounted for: 613 / 613
Imaging cases accounted for: 1350 / 1350

Patient-level split validation passed.


### Preprocessing Pipeline Overview

```text
                         BraTS 2024 GLI Dataset
                                  │
                                  ▼
                         1,350 imaging cases
                           613 subjects
                                  │
                                  ▼
                         Dataset manifest
                    ┌─────────────┴─────────────┐
                    │ 4 MRI modalities / case  │
                    │ T1n · T1c · T2w · T2-FLAIR
                    │ + reference segmentation │
                    └─────────────┬─────────────┘
                                  │
                                  ▼
                       Patient-level partition
                                  │
              ┌───────────────────┼───────────────────┐
              ▼                   ▼                   ▼
           Training           Validation          Held-out test
             70%                 15%                  15%
        429 subjects          92 subjects          92 subjects
         936 scans             216 scans            198 scans
          (69.3%)               (16.0%)              (14.7%)
              │                   │                   │
              └───────────────────┴───────────────────┘
                                  │
                         No subject overlap
                                  │
                                  ▼
                         MONAI image loading
                                  │
                                  ▼
                  Four-channel multimodal MRI
                   [T1n, T1c, T2w, T2-FLAIR]
                                  │
                                  ▼
                       Spatial preprocessing
                                  │
                                  ▼
                      Intensity normalization
                                  │
                                  ▼
                    3D crop / patch sampling
                                  │
                                  ▼
                        Data augmentation
                         (training only)
                                  │
                                  ▼
                     MONAI Dataset / DataLoader
                                  │
                                  ▼
                     Model-ready 3D tensors
                                  │
                                  ▼
                  3D segmentation model
                       (next notebook)
```

## 9. Loading Multimodal MRI with MONAI

Before constructing the complete preprocessing pipeline, a single training
case is loaded using MONAI.

This allows the behaviour of MONAI's medical-image loading system to be
examined directly, including:

- loading the four MRI modalities as a multimodal image;
- loading the corresponding reference segmentation;
- representing image channels in a deep-learning-compatible format;
- retaining spatial metadata from the original NIfTI files.

The transforms will initially be applied individually so that each
preprocessing step can be understood and validated before they are
combined into the final pipeline.

In [13]:
# -------------------------------------------------------------
# LOAD ONE BRATS CASE USING MONAI
# -------------------------------------------------------------
#
# We begin with only the first training case.
#
# This lets us inspect what MONAI produces before applying the
# same processing pipeline to hundreds of cases.

from monai.transforms import LoadImaged


# Select one case from the training partition.
sample_case = train_cases[0]


print("Case ID:", sample_case["case_id"])
print("Subject ID:", sample_case["subject_id"])

print("\nMRI files:")

for image_path in sample_case["image"]:
    print(image_path)

print(
    "\nSegmentation:",
    sample_case["label"]
)


# -------------------------------------------------------------
# CREATE THE MONAI IMAGE-LOADING TRANSFORM
# -------------------------------------------------------------
#
# LoadImaged is the dictionary-based version of MONAI's image
# loader.
#
# We tell it to load the values stored under the dictionary
# keys "image" and "label".

load_transform = LoadImaged(
    keys=["image", "label"]
)


# -------------------------------------------------------------
# APPLY THE TRANSFORM
# -------------------------------------------------------------

loaded_case = load_transform(sample_case)

Case ID: BraTS-GLI-00006-100
Subject ID: BraTS-GLI-00006

MRI files:
/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00006-100/BraTS-GLI-00006-100-t1n.nii.gz
/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00006-100/BraTS-GLI-00006-100-t1c.nii.gz
/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00006-100/BraTS-GLI-00006-100-t2w.nii.gz
/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00006-100/BraTS-GLI-00006-100-t2f.nii.gz

Segmentation: /home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00006-100/BraTS-GLI-00006-100-seg.nii.gz


## 10. Inspecting the MONAI-Loaded Case

The selected BraTS case has now been loaded from its NIfTI files using
MONAI.

Before applying additional preprocessing, the loaded data are inspected
to determine:

- the object types produced by MONAI;
- the dimensions of the multimodal MRI and segmentation;
- how the four MRI modalities are represented as channels;
- the numerical data types;
- whether spatial metadata from the NIfTI files has been retained.

This provides a baseline representation against which subsequent
preprocessing transformations can be checked.

In [14]:
# -------------------------------------------------------------
# INSPECT THE MONAI-LOADED CASE
# -------------------------------------------------------------
#
# The NIfTI files have now been loaded from disk.
#
# We inspect the resulting objects before applying any further
# preprocessing so that we understand exactly what MONAI has
# created.

print(
    "Loaded case keys:",
    loaded_case.keys()
)


print("\n--- MRI IMAGE ---")

print(
    "Object type:",
    type(loaded_case["image"])
)

print(
    "Shape:",
    loaded_case["image"].shape
)

print(
    "Data type:",
    loaded_case["image"].dtype
)


print("\n--- SEGMENTATION LABEL ---")

print(
    "Object type:",
    type(loaded_case["label"])
)

print(
    "Shape:",
    loaded_case["label"].shape
)

print(
    "Data type:",
    loaded_case["label"].dtype
)

Loaded case keys: dict_keys(['case_id', 'image', 'label', 'subject_id'])

--- MRI IMAGE ---
Object type: <class 'monai.data.meta_tensor.MetaTensor'>
Shape: torch.Size([4, 182, 218, 182])
Data type: torch.float32

--- SEGMENTATION LABEL ---
Object type: <class 'monai.data.meta_tensor.MetaTensor'>
Shape: torch.Size([182, 218, 182])
Data type: torch.float32


### Spatial Metadata Preservation

MONAI represents the loaded NIfTI volumes as `MetaTensor` objects rather
than ordinary NumPy arrays.

A `MetaTensor` combines the numerical image data with associated
medical-imaging metadata. This allows spatial information to remain
associated with the image as preprocessing transformations are applied.

The metadata and affine transformation of the loaded MRI are inspected
before spatial preprocessing.

In [15]:
# -------------------------------------------------------------
# INSPECT MONAI SPATIAL METADATA
# -------------------------------------------------------------
#
# MetaTensor stores both the numerical voxel data and metadata
# describing the source medical image.
#
# We inspect some of this information before applying any
# spatial transformations.

image = loaded_case["image"]
label = loaded_case["label"]


print("--- MRI METADATA ---")

print(
    "Affine matrix:"
)

print(
    image.affine
)


print(
    "\nMetadata keys:"
)

print(
    image.meta.keys()
)


print(
    "\nSource filenames:"
)

print(
    image.meta.get("filename_or_obj")
)


print("\n--- SEGMENTATION METADATA ---")

print(
    "Affine matrix:"
)

print(
    label.affine
)

print(
    "\nSource filename:"
)

print(
    label.meta.get("filename_or_obj")
)

--- MRI METADATA ---
Affine matrix:
tensor([[  -1.,   -0.,    0.,   90.],
        [  -0.,    1.,   -0., -126.],
        [   0.,    0.,    1.,  -72.],
        [   0.,    0.,    0.,    1.]], dtype=torch.float64)

Metadata keys:
dict_keys(['sizeof_hdr', 'extents', 'session_error', 'dim_info', 'dim', 'intent_p1', 'intent_p2', 'intent_p3', 'intent_code', 'datatype', 'bitpix', 'slice_start', 'pixdim', 'vox_offset', 'scl_slope', 'scl_inter', 'slice_end', 'slice_code', 'xyzt_units', 'cal_max', 'cal_min', 'slice_duration', 'toffset', 'glmax', 'glmin', 'qform_code', 'sform_code', 'quatern_b', 'quatern_c', 'quatern_d', 'qoffset_x', 'qoffset_y', 'qoffset_z', 'srow_x', 'srow_y', 'srow_z', original_pixdim, affine, original_affine, 'as_closest_canonical', spatial_shape, space, original_channel_dim, 'filename_or_obj'])

Source filenames:
/home/corvus_splendens/datasets/brats2024-gli/training/training_data1_v2/BraTS-GLI-00006-100/BraTS-GLI-00006-100-t1n.nii.gz

--- SEGMENTATION METADATA ---
Affine matr

## 11. Ensuring Channel-First Formatting

Deep-learning frameworks generally represent 3D medical images using
a channel-first structure:

`[Channels, Depth, Height, Width]`

The four MRI modalities have already been stacked into four channels
by MONAI's loading transform.

The reference segmentation, however, is currently a three-dimensional
label map without an explicit channel dimension.

MONAI's `EnsureChannelFirstd` transform is used to standardize both
representations while preserving the underlying voxel data.

### Expected transformation

```text
                   BEFORE
                      │
         ┌────────────┴────────────┐
         ▼                         ▼
    MRI modalities            Segmentation
   [4,182,218,182]          [182,218,182]
         │                         │
         └────────────┬────────────┘
                      ▼
             EnsureChannelFirstd
                      │
         ┌────────────┴────────────┐
         ▼                         ▼
    MRI modalities            Segmentation
   [4,182,218,182]          [1,182,218,182]

                   AFTER
```

This transformation changes the tensor structure, not the anatomical
content or segmentation labels.

In [16]:
# -------------------------------------------------------------
# ENSURE CHANNEL-FIRST FORMATTING
# -------------------------------------------------------------
#
# A 3D segmentation model expects image tensors with an explicit
# channel dimension.
#
# The MRI already contains four channels because MONAI stacked
# the four modalities during loading.
#
# The segmentation currently has no channel dimension.
#
# EnsureChannelFirstd standardizes the representation using
# metadata retained by MONAI.

from monai.transforms import EnsureChannelFirstd


# -------------------------------------------------------------
# CREATE THE TRANSFORM
# -------------------------------------------------------------
#
# keys specifies which entries in the dictionary should be
# processed.
#
# The transform uses the image metadata to determine whether
# a channel dimension already exists.

channel_transform = EnsureChannelFirstd(
    keys=["image", "label"]
)


# -------------------------------------------------------------
# APPLY THE TRANSFORM
# -------------------------------------------------------------

channel_first_case = channel_transform(loaded_case)


# -------------------------------------------------------------
# COMPARE BEFORE AND AFTER
# -------------------------------------------------------------

print("--- BEFORE ---")

print(
    "MRI shape:",
    loaded_case["image"].shape
)

print(
    "Segmentation shape:",
    loaded_case["label"].shape
)


print("\n--- AFTER ---")

print(
    "MRI shape:",
    channel_first_case["image"].shape
)

print(
    "Segmentation shape:",
    channel_first_case["label"].shape
)

--- BEFORE ---
MRI shape: torch.Size([4, 182, 218, 182])
Segmentation shape: torch.Size([182, 218, 182])

--- AFTER ---
MRI shape: torch.Size([4, 182, 218, 182])
Segmentation shape: torch.Size([1, 182, 218, 182])


## 12. Spatial Orientation Standardization

The BraTS MRI volumes are stored with spatial orientation metadata
describing the relationship between voxel indices and anatomical space.

The representative MRI and segmentation currently use LAS
(Left–Anterior–Superior) orientation.

For consistency, both the multimodal MRI and reference segmentation
will be reoriented to RAS (Right–Anterior–Superior) using MONAI's
`Orientationd` transform.

The transformation updates voxel ordering and associated spatial
metadata without changing the underlying anatomy.

The MRI and segmentation must undergo the same spatial transformation
to preserve their alignment.

In [18]:
# -------------------------------------------------------------
# STANDARDIZE MRI AND SEGMENTATION ORIENTATION
# -------------------------------------------------------------
#
# The original BraTS volumes use LAS orientation.
#
# We standardize both the MRI and segmentation to RAS so that
# their spatial axes follow a consistent convention.
#
# MONAI updates the voxel arrangement and affine metadata
# together, preserving the physical locations of the anatomy.

from monai.transforms import Orientationd
from nibabel.orientations import aff2axcodes


# -------------------------------------------------------------
# CHECK ORIGINAL ORIENTATION
# -------------------------------------------------------------

print(
    "Original MRI orientation:",
    aff2axcodes(channel_first_case["image"].affine.numpy())
)

print(
    "Original label orientation:",
    aff2axcodes(channel_first_case["label"].affine.numpy())
)


# -------------------------------------------------------------
# CREATE THE ORIENTATION TRANSFORM
# -------------------------------------------------------------
#
# axcodes="RAS" specifies the desired anatomical orientation.
#
# Both the MRI and segmentation are transformed together.

orientation_transform = Orientationd(
    keys=["image", "label"],
    axcodes="RAS",
    labels=None
)


# -------------------------------------------------------------
# APPLY THE TRANSFORM
# -------------------------------------------------------------

oriented_case = orientation_transform(channel_first_case)


# -------------------------------------------------------------
# INSPECT THE RESULT
# -------------------------------------------------------------

print("\n--- AFTER REORIENTATION ---")

print(
    "MRI shape:",
    oriented_case["image"].shape
)

print(
    "Segmentation shape:",
    oriented_case["label"].shape
)

print(
    "MRI orientation:",
    aff2axcodes(oriented_case["image"].affine.numpy())
)

print(
    "Segmentation orientation:",
    aff2axcodes(oriented_case["label"].affine.numpy())
)

print(
    "\nMRI and segmentation affines match:",
    torch.allclose(
        oriented_case["image"].affine,
        oriented_case["label"].affine
    )
)

Original MRI orientation: ('L', 'A', 'S')
Original label orientation: ('L', 'A', 'S')

--- AFTER REORIENTATION ---
MRI shape: torch.Size([4, 182, 218, 182])
Segmentation shape: torch.Size([1, 182, 218, 182])
MRI orientation: ('R', 'A', 'S')
Segmentation orientation: ('R', 'A', 'S')

MRI and segmentation affines match: True


## 13. Foreground Intensity Normalization

MRI intensity values vary across imaging modalities and acquisitions.

To provide more consistent numerical inputs for model training,
foreground z-score normalization is applied independently to each
MRI channel.

For each modality:

1. Identify foreground voxels using a nonzero-intensity mask.
2. Calculate the foreground mean and standard deviation.
3. Normalize the foreground intensities.
4. Preserve the zero-valued background.

MONAI's `NormalizeIntensityd` transform provides this functionality.

The segmentation labels are excluded from intensity normalization,
because their numerical values represent anatomical classes rather
than MRI signal intensities.

In [19]:
# -------------------------------------------------------------
# NORMALIZE MRI INTENSITIES
# -------------------------------------------------------------
#
# Normalize each MRI modality independently.
#
# nonzero=True:
#   Calculate normalization statistics using only nonzero voxels,
#   avoiding the large zero-valued background.
#
# channel_wise=True:
#   Calculate separate mean and standard deviation values for
#   T1n, T1c, T2w and T2-FLAIR.
#
# Only "image" is transformed. Segmentation labels are unchanged.

from monai.transforms import NormalizeIntensityd


normalization_transform = NormalizeIntensityd(
    keys=["image"],
    nonzero=True,
    channel_wise=True
)


# Apply normalization after spatial reorientation.
normalized_case = normalization_transform(oriented_case)


# -------------------------------------------------------------
# INSPECT EACH MRI CHANNEL
# -------------------------------------------------------------

modality_names = [
    "T1n",
    "T1c",
    "T2w",
    "T2-FLAIR"
]

for channel_index, modality_name in enumerate(modality_names):

    # Extract one normalized MRI modality.
    volume = normalized_case["image"][channel_index]

    # Select nonzero foreground voxels.
    foreground = volume[volume != 0]

    print(f"\n--- {modality_name} ---")
    print("Foreground mean:", foreground.mean().item())
    print("Foreground std:", foreground.std(unbiased=False).item())
    print("Background voxels:", (volume == 0).sum().item())


# -------------------------------------------------------------
# VERIFY SEGMENTATION WAS NOT MODIFIED
# -------------------------------------------------------------

print("\nSegmentation unchanged:")

print(
    torch.equal(
        oriented_case["label"],
        normalized_case["label"]
    )
)


--- T1n ---
Foreground mean: -1.5744002723749873e-07
Foreground std: 1.0
Background voxels: 5757181

--- T1c ---
Foreground mean: 6.93803556828243e-08
Foreground std: 0.9999999403953552
Background voxels: 5757181

--- T2w ---
Foreground mean: -7.204882734868079e-08
Foreground std: 1.0
Background voxels: 5757181

--- T2-FLAIR ---
Foreground mean: 4.1363173863828706e-07
Foreground std: 1.0
Background voxels: 5757245

Segmentation unchanged:
True


## 14. Investigating Spatial Cropping

The full BraTS MRI volumes contain substantial zero-valued background
outside the brain.

Removing unnecessary background may reduce computational requirements
and improve the efficiency of subsequent patch sampling.

Before applying spatial cropping, the nonzero foreground extent of
the representative MRI is measured.

The proposed foreground bounding box is then checked against the
reference segmentation to ensure that no labelled tumour voxels
would be excluded.

In [20]:
# -------------------------------------------------------------
# INVESTIGATE FOREGROUND BOUNDING BOX
# -------------------------------------------------------------
#
# Identify the spatial extent of all nonzero MRI voxels.
#
# This is an exploratory calculation. We are NOT cropping
# the image yet.
#
# We use all four modalities to define the foreground mask,
# avoiding dependence on any single MRI sequence.

image = normalized_case["image"]
label = normalized_case["label"]

# Combine the nonzero regions of all four MRI channels.
# The resulting mask has spatial shape [D, H, W].
foreground_mask = torch.any(image != 0, dim=0)

# Locate all foreground voxel coordinates.
foreground_coordinates = torch.nonzero(
    foreground_mask,
    as_tuple=False
)

# Determine the smallest bounding box containing the foreground.
# Add 1 to maximum coordinates because Python slice endpoints
# are exclusive.
minimum_coordinates = foreground_coordinates.min(dim=0).values
maximum_coordinates = foreground_coordinates.max(dim=0).values + 1

cropped_shape = maximum_coordinates - minimum_coordinates

print("Original spatial shape:", tuple(image.shape[1:]))
print("Foreground bounding box start:", minimum_coordinates.tolist())
print("Foreground bounding box end:", maximum_coordinates.tolist())
print("Proposed cropped shape:", cropped_shape.tolist())

# -------------------------------------------------------------
# CHECK SEGMENTATION COVERAGE
# -------------------------------------------------------------
#
# Confirm that every labelled tumour voxel lies within the
# proposed MRI foreground bounding box.

tumour_mask = label[0] != 0

tumour_outside_foreground = torch.logical_and(
    tumour_mask,
    ~foreground_mask
).sum().item()

print(
    "\nTumour voxels outside MRI foreground:",
    tumour_outside_foreground
)

Original spatial shape: (182, 218, 182)
Foreground bounding box start: [26, 16, 18]
Foreground bounding box end: [160, 196, 157]
Proposed cropped shape: [134, 180, 139]

Tumour voxels outside MRI foreground: 0


## 15. Foreground Cropping with MONAI

The representative BraTS MRI contains a substantial amount of
zero-valued background surrounding the brain.

A preliminary bounding-box analysis demonstrated that the spatial
dimensions could be reduced from 182 × 218 × 182 to
134 × 180 × 139 without excluding any labelled tumour voxels
in the examined case.

MONAI's `CropForegroundd` transform is used to crop the image
and segmentation together.

The combined nonzero foreground across all four MRI modalities
defines the cropping region.

A margin of zero is initially used to reproduce the previously
calculated bounding box exactly. Additional margins may be
considered during final pipeline design.

The resulting crop is validated to ensure that the segmentation
labels and spatial alignment are preserved.

In [21]:
# -------------------------------------------------------------
# APPLY FOREGROUND CROPPING
# -------------------------------------------------------------
#
# CropForegroundd identifies a foreground bounding box using
# a source image and applies the same crop to the specified keys.
#
# source_key="image":
#   Use the multimodal MRI to determine the cropping region.
#
# select_fn:
#   Select voxels with nonzero intensity.
#
# channel_indices:
#   Use all four MRI channels when determining the foreground.
#
# margin=0:
#   Do not add additional voxels around the bounding box.
#
# allow_smaller=True:
#   Keep the crop within the available image boundaries.
#
# k_divisible=1:
#   Do not enlarge the crop to enforce divisible dimensions.

from monai.transforms import CropForegroundd


crop_transform = CropForegroundd(
    keys=["image", "label"],
    source_key="image",
    select_fn=lambda x: x != 0,
    channel_indices=[0, 1, 2, 3],
    margin=0,
    allow_smaller=True,
    k_divisible=1
)


# -------------------------------------------------------------
# APPLY THE TRANSFORM
# -------------------------------------------------------------

cropped_case = crop_transform(normalized_case)


# -------------------------------------------------------------
# COMPARE ORIGINAL AND CROPPED DIMENSIONS
# -------------------------------------------------------------

print("--- BEFORE CROPPING ---")

print(
    "MRI shape:",
    normalized_case["image"].shape
)

print(
    "Segmentation shape:",
    normalized_case["label"].shape
)


print("\n--- AFTER CROPPING ---")

print(
    "MRI shape:",
    cropped_case["image"].shape
)

print(
    "Segmentation shape:",
    cropped_case["label"].shape
)


# -------------------------------------------------------------
# VERIFY SEGMENTATION PRESERVATION
# -------------------------------------------------------------
#
# Compare the number of voxels belonging to each segmentation
# label before and after cropping.
#
# If the counts match, no labelled voxels have been removed.

original_labels, original_counts = torch.unique(
    normalized_case["label"],
    return_counts=True
)

cropped_labels, cropped_counts = torch.unique(
    cropped_case["label"],
    return_counts=True
)

print("\n--- SEGMENTATION LABEL COUNTS ---")

for label_value in original_labels:

    original_count = (
        normalized_case["label"] == label_value
    ).sum().item()

    cropped_count = (
        cropped_case["label"] == label_value
    ).sum().item()

    print(
        f"Label {int(label_value.item())}: "
        f"before={original_count}, "
        f"after={cropped_count}"
    )


# -------------------------------------------------------------
# CHECK SPATIAL ALIGNMENT
# -------------------------------------------------------------

print(
    "\nMRI and segmentation affines match:",
    torch.allclose(
        cropped_case["image"].affine,
        cropped_case["label"].affine
    )
)

--- BEFORE CROPPING ---
MRI shape: torch.Size([4, 182, 218, 182])
Segmentation shape: torch.Size([1, 182, 218, 182])

--- AFTER CROPPING ---
MRI shape: torch.Size([4, 134, 180, 139])
Segmentation shape: torch.Size([1, 134, 180, 139])

--- SEGMENTATION LABEL COUNTS ---
Label 0: before=7181051, after=3312699
Label 2: before=17907, after=17907
Label 4: before=22074, after=22074

MRI and segmentation affines match: True


## 16. Investigating Tumour Extent for Patch Sampling

Following foreground cropping, the spatial extent of the reference
tumour segmentation is examined.

The tumour bounding box is calculated using all nonzero segmentation
labels.

This provides an estimate of tumour dimensions and helps guide the
selection of an appropriate 3D training patch size.

A candidate patch size of 96 × 96 × 96 voxels is considered initially,
subject to further assessment of tumour coverage, GPU memory
requirements, and model performance.

In [22]:
# -------------------------------------------------------------
# MEASURE TUMOUR SPATIAL EXTENT
# -------------------------------------------------------------
#
# Identify the smallest rectangular bounding box containing
# all nonzero tumour segmentation labels.
#
# We use the cropped segmentation because this is the spatial
# representation that will be passed to subsequent preprocessing.
#
# Note: This examines one case only. We will need to evaluate
# tumour sizes across the dataset before finalizing patch size.

segmentation = cropped_case["label"][0]


# -------------------------------------------------------------
# IDENTIFY TUMOUR VOXELS
# -------------------------------------------------------------
#
# Any nonzero segmentation label is considered part of the
# annotated tumour for this bounding-box calculation.

tumour_mask = segmentation != 0

tumour_coordinates = torch.nonzero(
    tumour_mask,
    as_tuple=False
)


# -------------------------------------------------------------
# CALCULATE TUMOUR BOUNDING BOX
# -------------------------------------------------------------

tumour_minimum = tumour_coordinates.min(dim=0).values

tumour_maximum = (
    tumour_coordinates.max(dim=0).values + 1
)

tumour_dimensions = tumour_maximum - tumour_minimum


# -------------------------------------------------------------
# DISPLAY RESULTS
# -------------------------------------------------------------

print(
    "Cropped MRI spatial shape:",
    tuple(cropped_case["image"].shape[1:])
)

print(
    "Tumour bounding box start:",
    tumour_minimum.tolist()
)

print(
    "Tumour bounding box end:",
    tumour_maximum.tolist()
)

print(
    "Tumour bounding box dimensions:",
    tumour_dimensions.tolist()
)


# -------------------------------------------------------------
# COMPARE WITH CANDIDATE PATCH SIZE
# -------------------------------------------------------------

candidate_patch_size = torch.tensor([96, 96, 96])

tumour_fits = torch.all(
    tumour_dimensions <= candidate_patch_size
).item()

print(
    "\nTumour bounding box fits within 96³ patch:",
    tumour_fits
)

Cropped MRI spatial shape: (134, 180, 139)
Tumour bounding box start: [22, 121, 50]
Tumour bounding box end: [88, 177, 100]
Tumour bounding box dimensions: [66, 56, 50]

Tumour bounding box fits within 96³ patch: True


## 17. Tumour-Aware 3D Patch Sampling

Training a 3D segmentation network on entire MRI volumes can require
substantial GPU memory.

A candidate patch size of 96 × 96 × 96 voxels is therefore selected
for initial investigation.

MONAI's `RandCropByPosNegLabeld` transform will be used to sample
training patches around positive and negative segmentation locations.

Positive locations correspond to nonzero segmentation labels.
Negative locations correspond to background labels.

The MRI foreground is used to restrict negative sampling primarily
to regions containing image information.

The initial positive-to-negative sampling ratio is 1:1.

This strategy is intended for training only. Validation and final
testing will use a separate deterministic evaluation workflow to
avoid introducing random sampling into performance assessment.

In [23]:
# -------------------------------------------------------------
# TUMOUR-AWARE 3D PATCH SAMPLING
# -------------------------------------------------------------
#
# RandCropByPosNegLabeld extracts fixed-size 3D patches.
#
# Positive sampling:
#   Centre a patch on a voxel with a nonzero segmentation label.
#
# Negative sampling:
#   Centre a patch on a voxel with a background label.
#
# image_key="image":
#   Use MRI intensity information to restrict negative sampling
#   to image foreground rather than empty surrounding space.
#
# image_threshold=0:
#   Negative candidate locations must have image intensity > 0
#   according to MONAI's image foreground criterion.
#
# num_samples=2:
#   Generate two sampled patches from the selected case.
#
# pos=1, neg=1:
#   Give positive and negative sampling equal probability.

from monai.transforms import RandCropByPosNegLabeld


patch_transform = RandCropByPosNegLabeld(
    keys=["image", "label"],
    label_key="label",
    spatial_size=(96, 96, 96),
    pos=1,
    neg=1,
    num_samples=2,
    image_key="image",
    image_threshold=0
)


# -------------------------------------------------------------
# APPLY PATCH SAMPLING
# -------------------------------------------------------------
#
# Unlike the previous transforms, this transform returns a
# list of dictionaries because num_samples=2.

sampled_patches = patch_transform(cropped_case)


# -------------------------------------------------------------
# INSPECT THE SAMPLED PATCHES
# -------------------------------------------------------------

print("Number of sampled patches:", len(sampled_patches))

for patch_index, patch in enumerate(sampled_patches):

    patch_image = patch["image"]
    patch_label = patch["label"]

    tumour_voxels = (patch_label != 0).sum().item()

    print(f"\n--- Patch {patch_index + 1} ---")

    print("MRI shape:", patch_image.shape)
    print("Segmentation shape:", patch_label.shape)
    print("Tumour voxels:", tumour_voxels)

    print(
        "Segmentation labels:",
        torch.unique(patch_label).tolist()
    )

Number of sampled patches: 2

--- Patch 1 ---
MRI shape: torch.Size([4, 96, 96, 96])
Segmentation shape: torch.Size([1, 96, 96, 96])
Tumour voxels: 0
Segmentation labels: [0.0]

--- Patch 2 ---
MRI shape: torch.Size([4, 96, 96, 96])
Segmentation shape: torch.Size([1, 96, 96, 96])
Tumour voxels: 28740
Segmentation labels: [0.0, 2.0, 4.0]


## 18. Inspecting Sampled Patch Composition

The initial random sampling experiment generated two patches:

- One patch containing background only.
- One patch containing a subset of the annotated tumour.

The distribution of segmentation labels within each patch is examined
to understand how random sampling affects the representation of
individual tumour compartments.

This is particularly relevant for class imbalance, as some tumour
compartments may occupy substantially fewer voxels than others.

The findings will help guide the eventual training sampling strategy.

In [24]:
# -------------------------------------------------------------
# INSPECT SEGMENTATION LABEL DISTRIBUTION IN SAMPLED PATCHES
# -------------------------------------------------------------
#
# Count how many voxels belong to each segmentation label
# within each sampled 3D patch.
#
# This helps us understand the class distribution produced
# by tumour-aware random sampling.

for patch_index, patch in enumerate(sampled_patches):

    patch_label = patch["label"]

    # Find the unique label values and their voxel counts.
    unique_labels, label_counts = torch.unique(
        patch_label,
        return_counts=True
    )

    print(f"\n--- Patch {patch_index + 1} ---")

    # Display the count for every label present in the patch.
    for label_value, voxel_count in zip(
        unique_labels,
        label_counts
    ):

        print(
            f"Label {int(label_value.item())}: "
            f"{voxel_count.item():,} voxels"
        )

    # Calculate the proportion of the patch occupied by tumour.
    total_voxels = patch_label.numel()

    tumour_voxels = (patch_label != 0).sum().item()

    tumour_percentage = (
        tumour_voxels / total_voxels
    ) * 100

    print(
        f"Tumour fraction: {tumour_percentage:.2f}%"
    )


--- Patch 1 ---
Label 0: 884,736 voxels
Tumour fraction: 0.00%

--- Patch 2 ---
Label 0: 855,996 voxels
Label 2: 16,596 voxels
Label 4: 12,144 voxels
Tumour fraction: 3.25%



### 18.1 Results and Interpretation of 3D Patch Sampling

MONAI's `RandCropByPosNegLabeld` successfully generated two 3D patches from the representative BraTS case using a candidate patch size of `96 × 96 × 96` voxels.

Each patch contained:

- **MRI input:** `[4, 96, 96, 96]`, representing T1n, T1c, T2w and T2-FLAIR.
- **Segmentation:** `[1, 96, 96, 96]`, containing the corresponding reference labels.

#### Segmentation Label Distribution

| Segmentation label | Patch 1 | Patch 2 |
|---|---:|---:|
| Background (0) | 884,736 | 855,996 |
| Label 2 | 0 | 16,596 |
| Label 4 | 0 | 12,144 |
| **Total tumour voxels** | **0** | **28,740** |
| **Tumour fraction** | **0.00%** | **3.25%** |

#### Interpretation

**Patch 1 — Tumour-free patch**

The first patch contained only background-labelled voxels. Such patches provide negative training examples, helping the model learn to distinguish tumour tissue from normal brain structures and other non-tumour regions.

**Patch 2 — Tumour-containing patch**

The second patch contained both tumour labels present in the representative case. However, tumour voxels occupied only 3.25% of the patch, demonstrating substantial class imbalance even after tumour-aware sampling.

Compared with the complete cropped segmentation:

| Label | Full segmentation | Patch 2 | Proportion captured |
|---|---:|---:|---:|
| 2 | 17,907 | 16,596 | 92.7% |
| 4 | 22,074 | 12,144 | 55.0% |
| **Combined tumour** | **39,981** | **28,740** | **71.9%** |

Although the tumour bounding box measured only `66 × 56 × 50` voxels and could geometrically fit within a `96 × 96 × 96` patch, the sampled tumour-containing patch captured only 71.9% of its annotated volume.

This illustrates that sampling around a positive tumour voxel does not guarantee inclusion of the entire tumour.

#### Implications for Model Training

1. **Positive and negative sampling:** Sampling around both tumour and background voxels provides varied training examples and may help reduce false-positive predictions.

2. **Class imbalance:** Background remains the dominant class. Voxel-wise accuracy alone would therefore be misleading. Region-based Dice metrics and suitable segmentation loss functions will be required.

3. **Tumour compartment representation:** Different patches may contain substantially different proportions of individual tumour labels. Repeated random sampling can expose the model to different tumour regions and boundaries.

4. **Patch size:** The initial `96 × 96 × 96` patch size appears feasible for this representative case, but suitability must be assessed across the wider dataset and against available GPU memory.

5. **Sampling probabilities:** A 1:1 positive-to-negative sampling ratio controls the selection of patch-centre locations. It does not guarantee equal numbers of tumour-containing and tumour-free patches, or balanced representation of individual tumour compartments.

#### Conclusion

The initial patch-sampling experiment demonstrates that MONAI can generate correctly aligned, fixed-size multimodal MRI and segmentation patches containing different amounts of tumour tissue.

The findings support further investigation of tumour-aware patch sampling for training a 3D segmentation model.

**These observations are based on two sampled patches from one case and do not yet establish the optimal sampling strategy for the complete BraTS dataset.**


## 19. Three-Dimensional Data Augmentation

Data augmentation introduces controlled variation into the training
dataset to improve model generalization.

Spatial transformations must be applied consistently to both MRI
images and reference segmentations to preserve voxel-wise alignment.

Intensity transformations are applied only to MRI images, as
segmentation labels represent discrete tissue classes.

The initial augmentation experiment uses MONAI's `RandFlipd`
transform to demonstrate synchronized spatial augmentation.

Augmentation will be restricted to the training pipeline and
excluded from deterministic validation and test preprocessing.

In [25]:
# -------------------------------------------------------------
# EXPERIMENT WITH RANDOM SPATIAL FLIPPING
# -------------------------------------------------------------
#
# RandFlipd randomly reverses voxel ordering along a chosen
# spatial axis.
#
# keys=["image", "label"]:
#   Apply the SAME random transformation to the MRI and
#   segmentation so that they remain spatially aligned.
#
# prob=1.0:
#   Force the flip for this demonstration.
#   We will use a lower probability during actual training.
#
# spatial_axis=0:
#   Flip along the first spatial dimension.
#
# Since we have reoriented the MRI to RAS, this corresponds
# to reversing the left-right voxel axis.

from monai.transforms import RandFlipd


# Select the tumour-containing patch for demonstration.
original_patch = sampled_patches[1]


# Create a deterministic demonstration of a random flip.
flip_transform = RandFlipd(
    keys=["image", "label"],
    prob=1.0,
    spatial_axis=0
)


# Apply the transformation.
flipped_patch = flip_transform(original_patch)


# -------------------------------------------------------------
# COMPARE SHAPES
# -------------------------------------------------------------

print("--- ORIGINAL PATCH ---")

print("MRI shape:", original_patch["image"].shape)
print("Label shape:", original_patch["label"].shape)


print("\n--- FLIPPED PATCH ---")

print("MRI shape:", flipped_patch["image"].shape)
print("Label shape:", flipped_patch["label"].shape)


# -------------------------------------------------------------
# VERIFY THE SPATIAL TRANSFORMATION
# -------------------------------------------------------------
#
# torch.flip manually reverses the same spatial axis.
#
# The image has shape [C, D, H, W], so spatial axis 0
# corresponds to tensor dimension 1.

expected_flipped_image = torch.flip(
    original_patch["image"],
    dims=[1]
)

expected_flipped_label = torch.flip(
    original_patch["label"],
    dims=[1]
)


print(
    "\nMRI flip correct:",
    torch.equal(
        flipped_patch["image"],
        expected_flipped_image
    )
)

print(
    "Segmentation flip correct:",
    torch.equal(
        flipped_patch["label"],
        expected_flipped_label
    )
)


# -------------------------------------------------------------
# VERIFY TUMOUR LABEL COUNTS
# -------------------------------------------------------------
#
# A flip changes voxel positions but should not change
# the number of voxels belonging to each class.

for label_value in [0, 2, 4]:

    original_count = (
        original_patch["label"] == label_value
    ).sum().item()

    flipped_count = (
        flipped_patch["label"] == label_value
    ).sum().item()

    print(
        f"Label {label_value}: "
        f"original={original_count:,}, "
        f"flipped={flipped_count:,}"
    )

--- ORIGINAL PATCH ---
MRI shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])

--- FLIPPED PATCH ---
MRI shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])

MRI flip correct: True
Segmentation flip correct: True
Label 0: original=855,996, flipped=855,996
Label 2: original=16,596, flipped=16,596
Label 4: original=12,144, flipped=12,144



### 19.1 Results and Validation of Spatial Augmentation

MONAI's `RandFlipd` transform was applied to a tumour-containing
3D MRI patch using a forced flip probability of 1.0 along the
first spatial axis.

The same transformation was applied to the four-channel MRI and
its corresponding reference segmentation.

#### Results

| Validation criterion | Result |
|---|---|
| Original MRI shape | `[4, 96, 96, 96]` |
| Flipped MRI shape | `[4, 96, 96, 96]` |
| Original segmentation shape | `[1, 96, 96, 96]` |
| Flipped segmentation shape | `[1, 96, 96, 96]` |
| MRI flip matches expected result | True |
| Segmentation flip matches expected result | True |

#### Segmentation Label Preservation

| Label | Before flipping | After flipping |
|---|---:|---:|
| 0 | 855,996 | 855,996 |
| 2 | 16,596 | 16,596 |
| 4 | 12,144 | 12,144 |

All segmentation label counts remained unchanged.

#### Interpretation

The experiment confirms that MONAI can apply synchronized spatial
augmentation to multimodal MRI data and corresponding segmentation
labels.

Flipping changes the spatial arrangement of voxels without altering
the segmentation class distribution.

Because the MRI was previously standardized to RAS orientation,
flipping the first spatial axis corresponds to left-right reversal
in voxel space.

Left-right flipping may help the model learn tumour appearances
across both hemispheres. However, its clinical appropriateness and
effect on generalization should be evaluated rather than assumed.

#### Conclusion

The initial spatial augmentation experiment was successful.

Random spatial flipping can be incorporated into the training
preprocessing pipeline, with an appropriate probability rather
than the forced probability used for this demonstration.

Spatial augmentation will be excluded from deterministic
validation and test preprocessing.



## 20. Ninety-Degree Spatial Rotation Augmentation

Random spatial rotations can introduce additional anatomical
variation into the training dataset.

MONAI's `RandRotate90d` transform performs rotations in
multiples of 90 degrees.

Unlike arbitrary-angle rotations, these transformations
rearrange voxel positions without requiring interpolation.

For this experiment, a single 90-degree rotation is applied
to the MRI and segmentation together.

The experiment verifies:

- Preservation of MRI and segmentation dimensions.
- Synchronized transformation of image and label.
- Preservation of segmentation class counts.
- Agreement with the equivalent PyTorch rotation operation.

A forced rotation probability of 1.0 is used for demonstration.
The eventual training pipeline may use a lower probability.

Large anatomical rotations are not automatically assumed to
be clinically appropriate and will be evaluated cautiously.


In [26]:
# -------------------------------------------------------------
# EXPERIMENT WITH 90-DEGREE SPATIAL ROTATION
# -------------------------------------------------------------
#
# RandRotate90d performs rotations in multiples of 90 degrees.
#
# keys=["image", "label"]:
#   Apply the same spatial rotation to both MRI and segmentation.
#
# prob=1.0:
#   Force the rotation for this demonstration.
#
# max_k=1:
#   Rotate exactly once by 90 degrees when the transform runs.
#
# spatial_axes=(1, 2):
#   Rotate within the plane formed by spatial axes 1 and 2.
#
# Our image has shape [C, D, H, W], so spatial axes (1, 2)
# correspond to tensor dimensions (2, 3).

from monai.transforms import RandRotate90d


# Use the same tumour-containing patch as the flip experiment.
original_patch = sampled_patches[1]


# -------------------------------------------------------------
# CREATE AND APPLY THE ROTATION TRANSFORM
# -------------------------------------------------------------

rotation_transform = RandRotate90d(
    keys=["image", "label"],
    prob=1.0,
    max_k=1,
    spatial_axes=(1, 2)
)

rotated_patch = rotation_transform(original_patch)


# -------------------------------------------------------------
# INSPECT TENSOR SHAPES
# -------------------------------------------------------------

print("--- ORIGINAL PATCH ---")

print("MRI shape:", original_patch["image"].shape)
print("Label shape:", original_patch["label"].shape)


print("\n--- ROTATED PATCH ---")

print("MRI shape:", rotated_patch["image"].shape)
print("Label shape:", rotated_patch["label"].shape)


# -------------------------------------------------------------
# VERIFY THE ROTATION
# -------------------------------------------------------------
#
# torch.rot90 performs an equivalent 90-degree rotation.
#
# k=1 specifies one quarter-turn.
# dims=(2, 3) specifies the corresponding tensor dimensions.

expected_rotated_image = torch.rot90(
    original_patch["image"],
    k=1,
    dims=(2, 3)
)

expected_rotated_label = torch.rot90(
    original_patch["label"],
    k=1,
    dims=(2, 3)
)


print(
    "\nMRI rotation correct:",
    torch.equal(
        rotated_patch["image"],
        expected_rotated_image
    )
)

print(
    "Segmentation rotation correct:",
    torch.equal(
        rotated_patch["label"],
        expected_rotated_label
    )
)


# -------------------------------------------------------------
# VERIFY SEGMENTATION LABEL PRESERVATION
# -------------------------------------------------------------

for label_value in [0, 2, 4]:

    original_count = (
        original_patch["label"] == label_value
    ).sum().item()

    rotated_count = (
        rotated_patch["label"] == label_value
    ).sum().item()

    print(
        f"Label {label_value}: "
        f"original={original_count:,}, "
        f"rotated={rotated_count:,}"
    )

--- ORIGINAL PATCH ---
MRI shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])

--- ROTATED PATCH ---
MRI shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])

MRI rotation correct: True
Segmentation rotation correct: True
Label 0: original=855,996, rotated=855,996
Label 2: original=16,596, rotated=16,596
Label 4: original=12,144, rotated=12,144



### 20.1 Results and Validation of 90-Degree Rotation

MONAI's `RandRotate90d` transform was applied to a representative
tumour-containing patch using a forced 90-degree rotation within
spatial axes `(1, 2)`.

The MRI and segmentation were rotated together to preserve
voxel-wise correspondence.

#### Validation Results

| Criterion | Result |
|---|---|
| Original MRI shape | `[4, 96, 96, 96]` |
| Rotated MRI shape | `[4, 96, 96, 96]` |
| Original segmentation shape | `[1, 96, 96, 96]` |
| Rotated segmentation shape | `[1, 96, 96, 96]` |
| MRI rotation matches `torch.rot90` | True |
| Segmentation rotation matches `torch.rot90` | True |

#### Segmentation Label Preservation

| Label | Original | Rotated |
|---|---:|---:|
| 0 | 855,996 | 855,996 |
| 2 | 16,596 | 16,596 |
| 4 | 12,144 | 12,144 |

All segmentation label counts were preserved.

#### Interpretation

The experiment demonstrates that 90-degree spatial rotations
can be applied consistently to multimodal MRI and segmentation
data without interpolation or loss of segmentation labels.

Unlike arbitrary-angle rotations, rotations in multiples of
90 degrees preserve the original voxel values.

However, mathematically valid spatial transformations are not
necessarily clinically representative. The anatomical plausibility
of large rotations should be considered when designing the final
augmentation strategy.

#### Conclusion

The rotation experiment was successful and supports the
technical feasibility of including 90-degree rotations in the
training augmentation pipeline.

The eventual choice of rotation axes and probabilities will
be guided by anatomical considerations and model performance.



## 21. MRI Intensity Augmentation

MRI signal intensities are not standardized across scanners,
acquisition protocols, or imaging institutions.

Intensity augmentation introduces controlled variation into
MRI signal values to help improve model robustness.

Unlike spatial augmentation, intensity augmentation does not
change voxel positions.

In this experiment, MONAI's `RandScaleIntensityd` is used
to modify MRI intensities while preserving the segmentation.

The experiment will verify that:

- MRI and segmentation shapes remain unchanged.
- MRI intensity values are modified.
- Segmentation labels remain identical.
- Spatial alignment is preserved.


In [27]:
# -------------------------------------------------------------
# EXPERIMENT WITH MRI INTENSITY SCALING
# -------------------------------------------------------------
#
# RandScaleIntensityd multiplies MRI intensities by a randomly
# selected scaling factor.
#
# keys=["image"]:
#   Apply the transformation only to MRI data.
#
# factors=0.10:
#   Sample a multiplicative factor between 0.90 and 1.10.
#
# prob=1.0:
#   Force augmentation for this demonstration.
#
# channel_wise=True:
#   Independently sample scaling factors for the four MRI
#   modalities.

from monai.transforms import RandScaleIntensityd


original_patch = sampled_patches[1]


intensity_transform = RandScaleIntensityd(
    keys=["image"],
    factors=0.10,
    prob=1.0,
    channel_wise=True
)


# Apply MRI-only intensity augmentation.
intensity_augmented_patch = intensity_transform(
    original_patch
)


# -------------------------------------------------------------
# VERIFY TENSOR SHAPES
# -------------------------------------------------------------

print("--- ORIGINAL PATCH ---")
print("MRI shape:", original_patch["image"].shape)
print("Label shape:", original_patch["label"].shape)

print("\n--- INTENSITY-AUGMENTED PATCH ---")
print("MRI shape:", intensity_augmented_patch["image"].shape)
print("Label shape:", intensity_augmented_patch["label"].shape)


# -------------------------------------------------------------
# VERIFY MRI INTENSITY CHANGES
# -------------------------------------------------------------

for channel_index, modality_name in enumerate(
    ["T1n", "T1c", "T2w", "T2-FLAIR"]
):

    original_image = original_patch["image"][channel_index]

    augmented_image = (
        intensity_augmented_patch["image"][channel_index]
    )

    maximum_difference = torch.max(
        torch.abs(augmented_image - original_image)
    ).item()

    print(
        f"{modality_name}: "
        f"maximum intensity difference = "
        f"{maximum_difference:.6f}"
    )


# -------------------------------------------------------------
# VERIFY SEGMENTATION IS UNCHANGED
# -------------------------------------------------------------

segmentation_unchanged = torch.equal(
    original_patch["label"],
    intensity_augmented_patch["label"]
)

print(
    "\nSegmentation unchanged:",
    segmentation_unchanged
)


# -------------------------------------------------------------
# VERIFY BACKGROUND REMAINS ZERO
# -------------------------------------------------------------
#
# Multiplicative scaling preserves exact zero values.
# This is useful because our MRI background is zero.

original_background = original_patch["image"] == 0

background_preserved = torch.all(
    intensity_augmented_patch["image"][original_background] == 0
).item()

print(
    "Zero-valued MRI background preserved:",
    background_preserved
)

--- ORIGINAL PATCH ---
MRI shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])

--- INTENSITY-AUGMENTED PATCH ---
MRI shape: torch.Size([4, 96, 96, 96])
Label shape: torch.Size([1, 96, 96, 96])
T1n: maximum intensity difference = 0.303831
T1c: maximum intensity difference = 0.762525
T2w: maximum intensity difference = 0.523323
T2-FLAIR: maximum intensity difference = 0.405666

Segmentation unchanged: True
Zero-valued MRI background preserved: True



### 21.1 Results and Validation of MRI Intensity Scaling

MONAI's `RandScaleIntensityd` was applied to a representative
four-channel MRI patch with a scaling range of ±10%.

Independent scaling factors were sampled for each MRI modality
using `channel_wise=True`.

The segmentation was excluded from the transformation.

#### Validation Results

| Criterion | Result |
|---|---|
| MRI shape preserved | True |
| Segmentation shape preserved | True |
| MRI intensity values modified | True |
| Segmentation unchanged | True |
| Zero-valued MRI background preserved | True |

#### Maximum Absolute Intensity Differences

| Modality | Maximum difference |
|---|---:|
| T1n | 0.303831 |
| T1c | 0.762525 |
| T2w | 0.523323 |
| T2-FLAIR | 0.405666 |

#### Interpretation

Intensity scaling successfully introduced controlled changes
to the MRI signal values without modifying the corresponding
segmentation labels.

The differences between modalities reflect independently
sampled intensity scaling factors.

Multiplicative scaling also preserved zero-valued MRI
background voxels, maintaining the distinction between
foreground signal and background padding.

This experiment demonstrates that MRI intensity augmentation
can introduce additional training variation without altering
the anatomical positions of voxels.

#### Conclusion

The initial MRI intensity scaling experiment was successful.

Channel-wise intensity scaling is a candidate augmentation
for the training pipeline. Its effect on segmentation
performance will require empirical evaluation.



## 22. MRI Intensity Shifting

Intensity shifting adds a randomly sampled offset to MRI
voxel intensities.

This transformation can simulate certain variations in MRI
signal intensity distributions.

Unlike multiplicative scaling, additive shifting changes
zero-valued voxels unless the background is explicitly
excluded or restored.

Because our preprocessing pipeline uses zero-valued
background voxels, the original background mask will be
preserved during this experiment.

The segmentation must remain unchanged.


In [28]:
# -------------------------------------------------------------
# EXPERIMENT WITH MRI INTENSITY SHIFTING
# -------------------------------------------------------------
#
# RandShiftIntensityd adds a randomly sampled offset to MRI
# intensities.
#
# offsets=0.10:
#   Sample an additive offset between -0.10 and +0.10.
#
# prob=1.0:
#   Force the transformation for demonstration.
#
# channel_wise=True:
#   Independently sample an offset for each MRI modality.
#
# IMPORTANT:
#   Additive shifts can turn background zeros into nonzero
#   values. We therefore restore the original background
#   after augmentation.

from monai.transforms import RandShiftIntensityd


original_patch = sampled_patches[1]


# Create the intensity shifting transform.
shift_transform = RandShiftIntensityd(
    keys=["image"],
    offsets=0.10,
    prob=1.0,
    channel_wise=True
)


# Apply intensity shifting to the MRI.
shifted_patch = shift_transform(original_patch)


# -------------------------------------------------------------
# RESTORE ZERO-VALUED BACKGROUND
# -------------------------------------------------------------

# Identify original zero-valued voxels for each modality.
background_mask = original_patch["image"] == 0

# Restore zeros without modifying the original patch.
shifted_patch["image"] = shifted_patch["image"].clone()

shifted_patch["image"][background_mask] = 0


# -------------------------------------------------------------
# VERIFY MRI INTENSITY CHANGES
# -------------------------------------------------------------

for channel_index, modality_name in enumerate(
    ["T1n", "T1c", "T2w", "T2-FLAIR"]
):

    original_image = original_patch["image"][channel_index]

    shifted_image = shifted_patch["image"][channel_index]

    maximum_difference = torch.max(
        torch.abs(shifted_image - original_image)
    ).item()

    print(
        f"{modality_name}: "
        f"maximum intensity difference = "
        f"{maximum_difference:.6f}"
    )


# -------------------------------------------------------------
# VERIFY SEGMENTATION IS UNCHANGED
# -------------------------------------------------------------

segmentation_unchanged = torch.equal(
    original_patch["label"],
    shifted_patch["label"]
)

print(
    "\nSegmentation unchanged:",
    segmentation_unchanged
)


# -------------------------------------------------------------
# VERIFY ZERO-VALUED BACKGROUND IS PRESERVED
# -------------------------------------------------------------

background_preserved = torch.all(
    shifted_patch["image"][background_mask] == 0
).item()

print(
    "Zero-valued MRI background preserved:",
    background_preserved
)


# -------------------------------------------------------------
# VERIFY TENSOR SHAPES
# -------------------------------------------------------------

print("\nOriginal MRI shape:", original_patch["image"].shape)
print("Shifted MRI shape:", shifted_patch["image"].shape)

print("Original label shape:", original_patch["label"].shape)
print("Shifted label shape:", shifted_patch["label"].shape)

T1n: maximum intensity difference = 0.025893
T1c: maximum intensity difference = 0.004583
T2w: maximum intensity difference = 0.060226
T2-FLAIR: maximum intensity difference = 0.059385

Segmentation unchanged: True
Zero-valued MRI background preserved: True

Original MRI shape: torch.Size([4, 96, 96, 96])
Shifted MRI shape: torch.Size([4, 96, 96, 96])
Original label shape: torch.Size([1, 96, 96, 96])
Shifted label shape: torch.Size([1, 96, 96, 96])



### 22.1 Results and Validation of MRI Intensity Shifting

MONAI's `RandShiftIntensityd` was applied to a representative
four-channel MRI patch using independently sampled intensity
offsets within ±0.10.

The original zero-valued MRI background was restored after
augmentation, and the segmentation was excluded from the
intensity transformation.

#### Validation Results

| Criterion | Result |
|---|---|
| MRI dimensions preserved | True |
| Segmentation dimensions preserved | True |
| MRI intensities modified | True |
| Segmentation unchanged | True |
| Zero-valued MRI background preserved | True |

#### Maximum Absolute Intensity Differences

| Modality | Maximum difference |
|---|---:|
| T1n | 0.025893 |
| T1c | 0.004583 |
| T2w | 0.060226 |
| T2-FLAIR | 0.059385 |

#### Interpretation

Intensity shifting successfully modified the MRI signal
values without changing voxel positions or segmentation
labels.

Different offsets were sampled for the four MRI modalities,
introducing variation in their normalized intensity
distributions.

Unlike multiplicative intensity scaling, additive shifting
would ordinarily change zero-valued background voxels.
Restoring the original background mask prevented this
unwanted effect.

The observed intensity differences were within the
configured augmentation range.

#### Methodological Considerations

Foreground z-score normalization already reduces global
intensity offsets and scaling differences between MRI scans.

Therefore, the additional benefit of intensity shifting
cannot be assumed.

The experiment demonstrates technical feasibility, but
inclusion in the final training pipeline should be guided
by validation performance.

#### Conclusion

MRI intensity shifting was successfully implemented and
validated on a representative patch.

It remains an optional candidate for the final training
augmentation strategy.
